In [22]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from math import radians, sin, cos, sqrt, asin

In [3]:
df = pd.read_csv("vuelos_kayak_data2.csv", sep=";", engine="python", encoding="utf-8")

In [4]:
df = df
df.head()

,fecha_vuelo,precio,aerolinea,hora_salida,hora_llegada,origen_iata,destino_iata,paradas
0,lun. 1/12,193.0,Ryanair,7:40,9:10,MAD,STN,directo
1,lun. 1/12,90.0,Ryanair,19:00,11:30\r\n+1,MAD,STN,2 escalas
2,lun. 1/12,207.0,easyJet,10:30,12:00,MAD,LTN,directo
3,lun. 1/12,268.0,Ryanair,14:45,16:15,MAD,STN,directo
4,lun. 1/12,287.0,easyJet,13:00,14:30,MAD,LGW,directo


### Map airports into corresponding cities

In [11]:
airport_to_city_full = {
    # Spain
    "MAD": "Madrid",
    # London airports -> London
    "LHR": "London", "LGW": "London", "LTN": "London",
    "STN": "London", "LCY": "London", "SEN": "London",
    # Paris
    "CDG": "Paris", "ORY": "Paris", "BVA": "Paris", "PAR": "Paris",
    # Berlin
    "BER": "Berlin", "SXF": "Berlin", "TXL": "Berlin",
    # Rome
    "FCO": "Rome", "CIA": "Rome", "ROM": "Rome",
    # Bucharest
    "OTP": "Bucharest", "BBU": "Bucharest",
    # Vienna
    "VIE": "Vienna",
    # Amsterdam
    "AMS": "Amsterdam",
    # Copenhagen
    "CPH": "Copenhagen",
    # Stockholm
    "ARN": "Stockholm", "BMA": "Stockholm",
    # Oslo
    "OSL": "Oslo",
    # Helsinki
    "HEL": "Helsinki",
    # Lisbon
    "LIS": "Lisbon",
    # Istanbul
    "IST": "Istanbul", "SAW": "Istanbul",
    # Zurich
    "ZRH": "Zurich",
    # Brussels
    "BRU": "Brussels",
    # Dublin
    "DUB": "Dublin",
    # Prague
    "PRG": "Prague",
    # Budapest
    "BUD": "Budapest",
    # Athens
    "ATH": "Athens",
    # Milan
    "MXP": "Milan", "LIN": "Milan", "BGY": "Milan", "MIL": "Milan",
    # Brussels
    "CRL": "Bruessels", 
    #Oslo
    "TRF": "Oslo" ,     
    # Stockholm
    "NYO": "Stockholm", "STO": "Stockholm",  "VST": "Stockholm"
}

# ------------- Map IATA -> city -------------
df["Origin"] = df["origen_iata"].map(airport_to_city_full)
df["Destination"] = df["destino_iata"].map(airport_to_city_full)

# ------------- Not mapped values -------------
df["Origin"] = df["Origin"].fillna("Unknown")
df["Destination"] = df["Destination"].fillna("Unknown")

df = df[
    (df["Origin"].notna()) &
    (df["Destination"].notna()) &
    (df["Origin"] != "Unknown") &
    (df["Destination"] != "Unknown")
].copy()

df.reset_index(drop=True, inplace=True)

df.head()

Unique unmapped origin IATA (count): 0
[]
Unique unmapped destination IATA (count): 0
[]


,fecha_vuelo,precio,aerolinea,hora_salida,hora_llegada,origen_iata,destino_iata,paradas,Origin,Destination
0,lun. 1/12,193.0,Ryanair,7:40,9:10,MAD,STN,directo,Madrid,London
1,lun. 1/12,90.0,Ryanair,19:00,11:30\r\n+1,MAD,STN,2 escalas,Madrid,London
2,lun. 1/12,207.0,easyJet,10:30,12:00,MAD,LTN,directo,Madrid,London
3,lun. 1/12,268.0,Ryanair,14:45,16:15,MAD,STN,directo,Madrid,London
4,lun. 1/12,287.0,easyJet,13:00,14:30,MAD,LGW,directo,Madrid,London


### Map date

In [13]:
fecha_col = "fecha_vuelo"
# Diccionario: español abreviado → inglés completo
weekday_map = {
    "lun": "Monday",
    "mar": "Tuesday",
    "mie": "Wednesday", "mié": "Wednesday",
    "jue": "Thursday",
    "vie": "Friday",
    "sab": "Saturday", "sáb": "Saturday",
    "dom": "Sunday"
}

def parse_fecha(raw):
    if pd.isna(raw):
        return pd.Series([np.nan, np.nan, np.nan, np.nan], 
                         index=["Year","Month","Day","Weekday"])

    s = str(raw).strip().lower()

    # 1. Separar día semana y parte numérica
    # Ej: "sáb. 1/12" → ["sáb.", "1/12"]
    try:
        weekday_es, date_part = s.split(" ", 1)
    except:
        # si no hay espacio, no podemos parsear
        return pd.Series([np.nan, np.nan, np.nan, np.nan], 
                         index=["Year","Month","Day","Weekday"])

    # 2. Limpiar el nombre del día
    weekday_es = weekday_es.replace(".", "").strip()

    # 3. Convertir día semana a inglés
    weekday_en = weekday_map.get(weekday_es, None)

    # 4. Extraer día y mes
    try:
        day, month = date_part.split("/")
        day = int(day)
        month = int(month)
    except:
        return pd.Series([np.nan, np.nan, np.nan, np.nan], 
                         index=["Year","Month","Day","Weekday"])

    # 5. Aplicar regla de año:
    # diciembre = 2025
    # otro mes = 2026
    if month == 12:
        year = 2025
    else:
        year = 2026

    return pd.Series([year, month, day, weekday_en], 
                     index=["Year","Month","Day","Weekday"])

# Aplicar al dataframe
df[["Year","Month","Day","Weekday"]] = df[fecha_col].apply(parse_fecha)



,fecha_vuelo,precio,aerolinea,hora_salida,hora_llegada,origen_iata,destino_iata,paradas,Origin,Destination,Year,Month,Day,Weekday
0,lun. 1/12,193.0,Ryanair,7:40,9:10,MAD,STN,directo,Madrid,London,2025.0,12.0,1.0,Monday
1,lun. 1/12,90.0,Ryanair,19:00,11:30\r\n+1,MAD,STN,2 escalas,Madrid,London,2025.0,12.0,1.0,Monday
2,lun. 1/12,207.0,easyJet,10:30,12:00,MAD,LTN,directo,Madrid,London,2025.0,12.0,1.0,Monday
3,lun. 1/12,268.0,Ryanair,14:45,16:15,MAD,STN,directo,Madrid,London,2025.0,12.0,1.0,Monday
4,lun. 1/12,287.0,easyJet,13:00,14:30,MAD,LGW,directo,Madrid,London,2025.0,12.0,1.0,Monday


### Number of stops

In [16]:

ef parse_stops(value):
    value = str(value).lower().strip()

    if "direct" in value:   # directo, directos, directo…
        return 0
    
    # Si contiene un número: "1 escala", "2 escalas", etc.
    for n in range(1, 10):  # por si hay 3,4… aunque normalmente es 0-2
        if str(n) in value:
            return n

    return np.nan  # por si aparece algo inesperado

df["Stops"] = df["paradas"].apply(parse_stops)
df.head()

,fecha_vuelo,precio,aerolinea,hora_salida,hora_llegada,origen_iata,destino_iata,paradas,Origin,Destination,Year,Month,Day,Weekday,Stops
0,lun. 1/12,193.0,Ryanair,7:40,9:10,MAD,STN,directo,Madrid,London,2025.0,12.0,1.0,Monday,0.0
1,lun. 1/12,90.0,Ryanair,19:00,11:30\r\n+1,MAD,STN,2 escalas,Madrid,London,2025.0,12.0,1.0,Monday,2.0
2,lun. 1/12,207.0,easyJet,10:30,12:00,MAD,LTN,directo,Madrid,London,2025.0,12.0,1.0,Monday,0.0
3,lun. 1/12,268.0,Ryanair,14:45,16:15,MAD,STN,directo,Madrid,London,2025.0,12.0,1.0,Monday,0.0
4,lun. 1/12,287.0,easyJet,13:00,14:30,MAD,LGW,directo,Madrid,London,2025.0,12.0,1.0,Monday,0.0


### Flight duration

In [21]:

city_timezone = {
    "Madrid": 1,       # CET
    "London": 0,       # GMT
    "Paris": 1,        # CET
    "Berlin": 1,       # CET
    "Rome": 1,         # CET
    "Bucharest": 2,    # EET
    "Vienna": 1,       # CET
    "Amsterdam": 1,    # CET
    "Copenhagen": 1,   # CET
    "Stockholm": 1,    # CET
    "Oslo": 1,         # CET
    "Helsinki": 2,     # EET
    "Lisbon": 0,       # WET
    "Istanbul": 3,     # TRT
    "Zurich": 1,       # CET
    "Brussels": 1,     # CET
    "Dublin": 0,       # GMT
    "Prague": 1,       # CET
    "Budapest": 1,     # CET
    "Athens": 2,       # EET
    "Milan": 1         # CET
}

def clean_time_field(t):
    if pd.isna(t):
        return None
    t = str(t).replace("\n", "").replace("\r", "").replace('"', '').strip()
    return t

def compute_duration(row):
    salida = clean_time_field(row["hora_salida"])
    llegada = clean_time_field(row["hora_llegada"])

    if salida is None or llegada is None:
        return np.nan

    # 1) Detect if the arrival is next day
    next_day = "+1" in llegada
    llegada = llegada.replace("+1", "").strip()

    try:
        t_s = datetime.strptime(salida, "%H:%M")
        t_l = datetime.strptime(llegada, "%H:%M")
    except:
        return np.nan

    if next_day:
        t_l += timedelta(hours=24)


    # TIMEZONE CORRECTION
    origin_city = row["Origin"]
    dest_city = row["Destination"]

    tz_origin = city_timezone.get(origin_city, None)
    tz_dest = city_timezone.get(dest_city, None)

    if tz_origin is None or tz_dest is None:
        return np.nan

    # Convert everything to UTC
    t_s_utc = t_s - timedelta(hours=tz_origin)
    t_l_utc = t_l - timedelta(hours=tz_dest)

    # Duration in minutes
    duration_minutes = (t_l_utc - t_s_utc).total_seconds() / 60


    return duration_minutes

df["Duration_minutes"] = df.apply(compute_duration, axis=1)
df["Duration_hours"] = df["Duration_minutes"] / 60

df.head()

,fecha_vuelo,precio,aerolinea,hora_salida,hora_llegada,origen_iata,destino_iata,paradas,Origin,Destination,Year,Month,Day,Weekday,Stops,Duration_minutes,Duration_hours
0,lun. 1/12,193.0,Ryanair,7:40,9:10,MAD,STN,directo,Madrid,London,2025.0,12.0,1.0,Monday,0.0,150.0,2.5
1,lun. 1/12,90.0,Ryanair,19:00,11:30\r\n+1,MAD,STN,2 escalas,Madrid,London,2025.0,12.0,1.0,Monday,2.0,1050.0,17.5
2,lun. 1/12,207.0,easyJet,10:30,12:00,MAD,LTN,directo,Madrid,London,2025.0,12.0,1.0,Monday,0.0,150.0,2.5
3,lun. 1/12,268.0,Ryanair,14:45,16:15,MAD,STN,directo,Madrid,London,2025.0,12.0,1.0,Monday,0.0,150.0,2.5
4,lun. 1/12,287.0,easyJet,13:00,14:30,MAD,LGW,directo,Madrid,London,2025.0,12.0,1.0,Monday,0.0,150.0,2.5


### City coordinates and distance

In [23]:
city_coords = {
    "Madrid": (40.4168, -3.7038),
    "London": (51.5074, -0.1278),
    "Paris": (48.8566, 2.3522),
    "Berlin": (52.5200, 13.4050),
    "Rome": (41.9028, 12.4964),
    "Bucharest": (44.4268, 26.1025),
    "Vienna": (48.2082, 16.3738),
    "Amsterdam": (52.3676, 4.9041),
    "Copenhagen": (55.6761, 12.5683),
    "Stockholm": (59.3293, 18.0686),
    "Oslo": (59.9139, 10.7522),
    "Helsinki": (60.1699, 24.9384),
    "Lisbon": (38.7223, -9.1393),
    "Istanbul": (41.0082, 28.9784),
    "Zurich": (47.3769, 8.5417),
    "Brussels": (50.8503, 4.3517),
    "Dublin": (53.3498, -6.2603),
    "Prague": (50.0755, 14.4378),
    "Budapest": (47.4979, 19.0402),
    "Athens": (37.9838, 23.7275),
    "Milan": (45.4642, 9.1900)
}

#Haversine function to calculate distance with coordinates
def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0  # Earth radius
    dlat = radians(lat2 - lat1)
    dlon = radians(lon2 - lon1)
    a = sin(dlat/2)**2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon/2)**2
    c = 2 * asin(sqrt(a))
    return R * c
    
def compute_distance(row):
    origin = row["Origin"]
    dest = row["Destination"]
    
    if origin not in city_coords or dest not in city_coords:
        return np.nan
    
    lat1, lon1 = city_coords[origin]
    lat2, lon2 = city_coords[dest]
    
    return haversine(lat1, lon1, lat2, lon2)

df["Distance_km"] = df.apply(compute_distance, axis=1)


In [24]:
df.head()


,fecha_vuelo,precio,aerolinea,hora_salida,hora_llegada,origen_iata,destino_iata,paradas,Origin,Destination,Year,Month,Day,Weekday,Stops,Duration_minutes,Duration_hours,Distance_km
0,lun. 1/12,193.0,Ryanair,7:40,9:10,MAD,STN,directo,Madrid,London,2025.0,12.0,1.0,Monday,0.0,150.0,2.5,1263.412055
1,lun. 1/12,90.0,Ryanair,19:00,11:30\r\n+1,MAD,STN,2 escalas,Madrid,London,2025.0,12.0,1.0,Monday,2.0,1050.0,17.5,1263.412055
2,lun. 1/12,207.0,easyJet,10:30,12:00,MAD,LTN,directo,Madrid,London,2025.0,12.0,1.0,Monday,0.0,150.0,2.5,1263.412055
3,lun. 1/12,268.0,Ryanair,14:45,16:15,MAD,STN,directo,Madrid,London,2025.0,12.0,1.0,Monday,0.0,150.0,2.5,1263.412055
4,lun. 1/12,287.0,easyJet,13:00,14:30,MAD,LGW,directo,Madrid,London,2025.0,12.0,1.0,Monday,0.0,150.0,2.5,1263.412055


In [27]:
# Rename columns
df_final = df.rename(columns={
    "precio": "Price",
    "aerolinea": "Airline",
    "hora_salida": "Departure_time",
    "hora_llegada": "Arrival_time",
    "origen_iata": "Origin_airport",
    "destino_iata": "Destination_airport"
})

df_final = df_final.drop(columns=["fecha_vuelo"])

#Round decimals
df_final["Distance_km"] = df_final["Distance_km"].round(3)
df_final["Duration_hours"] = df_final["Duration_hours"].round(3)
df_final["Duration_minutes"] = df_final["Duration_minutes"].round(0)

# 3. Orden opcional de columnas (más profesional)
ordered_cols = [
    "Year", "Month", "Day", "Weekday",
    "Origin", "Destination",
    "Origin_airport", "Destination_airport",
    "Departure_time", "Arrival_time",
    "Stops",
    "Duration_minutes", "Duration_hours",
    "Distance_km",
    "Price",
    "Airline"
]
df_final = df_final[[c for c in ordered_cols if c in df_final.columns]] 

# Save csv
df_final.to_csv("vuelos_final_dataset.csv", index=False, sep=";")


In [28]:
df_final.head(10)

,Year,Month,Day,Weekday,Origin,Destination,Origin_airport,Destination_airport,Departure_time,Arrival_time,Stops,Duration_minutes,Duration_hours,Distance_km,Price,Airline
0,2025.0,12.0,1.0,Monday,Madrid,London,MAD,STN,7:40,9:10,0.0,150.0,2.500,1263.412,193.0,Ryanair
1,2025.0,12.0,1.0,Monday,Madrid,London,MAD,STN,19:00,11:30\r\n+1,2.0,1050.0,17.500,1263.412,90.0,Ryanair
2,2025.0,12.0,1.0,Monday,Madrid,London,MAD,LTN,10:30,12:00,0.0,150.0,2.500,1263.412,207.0,easyJet
3,2025.0,12.0,1.0,Monday,Madrid,London,MAD,STN,14:45,16:15,0.0,150.0,2.500,1263.412,268.0,Ryanair
4,2025.0,12.0,1.0,Monday,Madrid,London,MAD,LGW,13:00,14:30,0.0,150.0,2.500,1263.412,287.0,easyJet
5,2025.0,12.0,1.0,Monday,Madrid,London,MAD,STN,6:10,7:40,0.0,150.0,2.500,1263.412,322.0,Ryanair
6,2025.0,12.0,1.0,Monday,Madrid,London,MAD,LGW,15:00,16:25,0.0,145.0,2.417,1263.412,293.0,Air Europa
7,2025.0,12.0,1.0,Monday,Madrid,London,MAD,STN,10:00,11:30,0.0,150.0,2.500,1263.412,317.0,Ryanair
8,2025.0,12.0,1.0,Monday,Madrid,London,MAD,LHR,13:20,14:55,0.0,155.0,2.583,1263.412,238.0,Iberia
9,2025.0,12.0,1.0,Monday,Madrid,London,MAD,STN,12:30,14:00,0.0,150.0,2.500,1263.412,312.0,Ryanair
